In [1]:
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sb

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/attila/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
vul = pd.read_csv("data/vulnerabilities.csv.gz")
exploitSignals = pd.read_csv("data/exploit_signals_2024.csv.gz")
exploitSignals2 = pd.read_csv("data/exploit_signals_2025.csv.gz")
knownExploits = pd.read_csv("data/known_exploited.csv.gz")
scoreBoard = pd.read_csv("data/scoreboard_2026.csv.gz")
weaknesses = pd.read_csv("data/weakness_types.csv.gz")
ref = pd.read_csv("data/references.csv.gz") 
affects = pd.read_csv("data/affected_products.csv.gz")

/var/folders/bz/6qk9ds4s1q52wqc85m4jh_qr0000gn/T/ipykernel_81780/725422424.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  affects = pd.read_csv("data/affected_products.csv.gz")


In [9]:
VulDetails = vul[vul['cvss3_score'] > 9.0].sort_values(by='cvss3_score', ascending=False).head(10)
VulDetails = VulDetails[['cve_id', 'cvss3_score', 'description']]
print (VulDetails)

                cve_id  cvss3_score  \
267571  CVE-2024-49668         10.0   
352037  CVE-2026-46798         10.0   
371820  CVE-2026-74705         10.0   
329298  CVE-2026-14812         10.0   
159660  CVE-2021-21960         10.0   
137068   CVE-2020-1889         10.0   
371913  CVE-2026-74843         10.0   
329094  CVE-2026-14560         10.0   
79766    CVE-2016-1044         10.0   
352082  CVE-2026-46846         10.0   

                                              description  
267571  Unrestricted Upload of File with Dangerous Typ...  
352037  Vulnerability in the Oracle WebCenter Sites pr...  
371820  In the Linux kernel, the following vulnerabili...  
329298  The Premium SEO WordPress plugin is malicious:...  
159660  A stack-based buffer overflow vulnerability ex...  
137068  A security feature bypass issue in WhatsApp De...  
371913  A vulnerability was determined in Wavlink WN53...  
329094  The teddy-bear-customize-addon WordPress plugi...  
79766   Adobe Reader and Acrob

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "ibm-granite/granite-4.0-micro" 
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto").to(device)
model.eval()

def ask(prompt, max_new_tokens=150):
    chat = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

W1107 16:28:24.986000 81780 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 322/322 [00:00<00:00, 5021.90it/s]


In [11]:
VulDetails = vul[vul['cvss3_score'] > 9.0].sort_values(by='cvss3_score', ascending=False).head(10)
VulDetails = VulDetails[['cve_id', 'cvss3_score', 'description']].copy()

def summarize_cve(desc):
    prompt = (
        "Summarize this vulnerability description in one plain-English sentence. "
        "State what is affected, what an attacker can do, and any key condition. "
        "Do not add information that is not in the text.\n\n"
        f"Description: {desc}"
    )
    return ask(prompt, max_new_tokens=80)

VulDetails['summary'] = VulDetails['description'].apply(summarize_cve)
print(VulDetails[['cve_id', 'cvss3_score', 'summary']].to_string(index=False))

        cve_id  cvss3_score                                                                                                                                                                                                                                                                                                                    summary
CVE-2024-49668         10.0                                                                                                             This vulnerability allows an attacker to upload a web shell to a web server, potentially gaining unauthorized access and control, affecting the Verbalize WP plugin version 1.0 and possibly earlier versions.
CVE-2026-46798         10.0                                                                          This vulnerability in Oracle WebCenter Sites allows unauthenticated attackers with network access to exploit the system, potentially leading to takeover, impacting confidentiality, integrity, and availability, wit